In [ ]:
repository_filter: list[str] = []
top_n_repos: int = 20

In [ ]:
import pandas as pd
from moderne_visualizations_misc.reusable.data_loader import read_data_table
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.simplefilter("ignore")

# df = read_data_table("../samples/test_quality_issues.csv")
df = read_data_table("../samples/v2/io.moderne.prethink.table.TestQualityIssues.csv")
df = qu.filter_repos(df, repository_filter)
df = qu.add_repo_short(df)

In [ ]:
if len(df) == 0:
    fig = qu.empty_figure()
else:
    severity_color_map = {
        "low": qu.SEVERITY_COLORS["LOW"],
        "medium": qu.SEVERITY_COLORS["MEDIUM"],
        "high": qu.SEVERITY_COLORS["HIGH"],
    }

    total_issues = len(df)
    total_repos = df["repoShort"].nunique()

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Worst Repositories",
            "Severity Distribution",
            "Issue Type Distribution",
            "Top 15 Fix Priorities",
        ),
        specs=[
            [{"type": "bar"}, {"type": "pie"}],
            [{"type": "bar"}, {"type": "table"}],
        ],
        vertical_spacing=0.15,
        horizontal_spacing=0.1,
    )

    # Panel 1: Worst Repositories (top-left)
    repo_counts = df.groupby("repoShort").size().nlargest(min(10, top_n_repos))
    high_counts = df[df["severity"] == "high"].groupby("repoShort").size()
    high_pct = (high_counts.reindex(repo_counts.index, fill_value=0) / repo_counts).fillna(0)

    colors = [
        f"rgb({int(200 * p + 55)}, {int(200 * (1 - p) + 55)}, 80)"
        for p in high_pct.values
    ]

    fig.add_trace(
        go.Bar(
            y=repo_counts.index.tolist()[::-1],
            x=repo_counts.values[::-1],
            orientation="h",
            marker_color=colors[::-1],
            hovertemplate="<b>%{y}</b><br>Issues: %{x}<extra></extra>",
            showlegend=False,
        ),
        row=1, col=1,
    )

    # Panel 2: Severity Distribution (top-right)
    sev_counts = df["severity"].value_counts()
    sev_labels = sev_counts.index.tolist()
    sev_colors = [severity_color_map.get(s, "#999") for s in sev_labels]

    fig.add_trace(
        go.Pie(
            labels=sev_labels,
            values=sev_counts.values,
            marker=dict(colors=sev_colors),
            textinfo="label+percent",
            hovertemplate="<b>%{label}</b><br>Count: %{value}<br>%{percent}<extra></extra>",
        ),
        row=1, col=2,
    )

    # Panel 3: Issue Type Distribution (bottom-left)
    type_sev = df.groupby(["issueType", "severity"]).size().reset_index(name="count")
    type_order = df.groupby("issueType").size().sort_values(ascending=False).index.tolist()

    worst_sev = df.groupby("issueType")["severity"].apply(
        lambda s: "high" if "high" in s.values else ("medium" if "medium" in s.values else "low")
    )
    type_colors = [severity_color_map.get(worst_sev.get(t, "low"), "#999") for t in type_order]

    type_counts = df.groupby("issueType").size().reindex(type_order, fill_value=0)

    fig.add_trace(
        go.Bar(
            x=type_order,
            y=type_counts.values,
            marker_color=type_colors,
            hovertemplate="<b>%{x}</b><br>Count: %{y}<extra></extra>",
            showlegend=False,
        ),
        row=2, col=1,
    )

    # Panel 4: Top 15 Fix Priorities (bottom-right)
    sev_rank = {"high": 3, "medium": 2, "low": 1}
    df["sevRank"] = df["severity"].map(sev_rank).fillna(0)
    priorities = df.sort_values(["sevRank", "issueType"], ascending=[False, True]).head(15)

    fig.add_trace(
        go.Table(
            header=dict(
                values=["Class", "Method", "Issue Type", "Severity"],
                fill_color="#f0f0f0",
                align="left",
                font=dict(size=11),
            ),
            cells=dict(
                values=[
                    priorities["className"].apply(qu.short_class).tolist(),
                    priorities["methodName"].fillna("").tolist(),
                    priorities["issueType"].tolist(),
                    priorities["severity"].tolist(),
                ],
                fill_color="white",
                align="left",
                font=dict(size=10),
            ),
        ),
        row=2, col=2,
    )

    fig.update_layout(
        title=f"Test Quality Dashboard — {total_issues} issues across {total_repos} repositories",
        height=900,
        margin=dict(l=60, r=50, t=80, b=60),
        showlegend=False,
    )
    fig.update_xaxes(tickangle=-45, row=2, col=1)

fig.show()